In [ ]:
from advsecurenet.computer_vision.image_classification.attacks.gradient_based import FGSM, LOTS, PGD
from advsecurenet.shared.types.configs.attack_configs import (
    FgsmAttackConfig,
    LotsAttackConfig,
    PgdAttackConfig,
)
from advsecurenet.shared.types.configs.attack_configs.attacker_config import (
    AttackerConfig,
)
from advsecurenet.models.model_factory import ModelFactory
from advsecurenet.datasets.dataset_factory import DatasetFactory
from advsecurenet.computer_vision.image_classification.attacks.attacker import Attacker
from advsecurenet.dataloader.data_loader_factory import DataLoaderFactory
from advsecurenet.shared.types.configs.preprocess_config import (
    PreprocessConfig,
    PreprocessStep,
)
from advsecurenet.shared.types.configs.device_config import DeviceConfig
from advsecurenet.utils.adversarial_target_generator import AdversarialTargetGenerator
from advsecurenet.datasets.targeted_adv_dataset import AdversarialDataset
from advsecurenet.computer_vision.image_classification.defenses.adversarial_training import AdversarialTraining
from advsecurenet.shared.types.configs.defense_configs.adversarial_training_config import (
    AdversarialTrainingConfig,
)
from advsecurenet.shared.types.configs import TrainConfig
from advsecurenet.shared.types.configs.train_config import (
    ModelConfig,
    TrainingProcessConfig
)

from advnet_common.types.configs.base import (
    OptimizationBase,
    CheckpointBase,
    FinalModelBase,
)

In [ ]:
# Initialize ResNet18 model for CIFAR-10 classification
model = ModelFactory.create_model(
    model_name="resnet18", architecture={"num_classes": 10}, pretrained=True
)

import torch.nn as nn

# Access the underlying model
if hasattr(model, 'model'):
    base_model = model.model
else:
    base_model = model

# Ensure the final layer has exactly 10 classes for CIFAR-10
if hasattr(base_model, 'fc'):
    in_features = base_model.fc.in_features
    base_model.fc = nn.Linear(in_features, 10)
    print(f"Final layer configured with 10 classes (input features: {in_features})")

# Verify model configuration
if hasattr(model, 'model') and hasattr(model.model, 'fc'):
    fc_layer = model.model.fc
elif hasattr(model, 'fc'):
    fc_layer = model.fc
else:
    fc_layer = None

if fc_layer:
    print(f"Model output classes: {fc_layer.out_features}")
    if fc_layer.out_features == 10:
        print("Model correctly configured for CIFAR-10")
    else:
        print(f"Warning: Model has {fc_layer.out_features} classes instead of 10")

In [ ]:
# Configure data preprocessing for CIFAR-10
preprocess_config = PreprocessConfig(
    steps=[
        PreprocessStep(name="Resize", params={"size": 32}),
        PreprocessStep(name="CenterCrop", params={"size": 32}),
        PreprocessStep(name="ToTensor"),
        PreprocessStep(
            name="ToDtype", params={"dtype": "torch.float32", "scale": True}
        ),
        PreprocessStep(
            name="Normalize",
            params={"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},
        ),
    ]
)

# Load CIFAR-10 dataset with preprocessing
dataset = DatasetFactory.load_dataset(
    dataset_name="cifar10", preprocessing=preprocess_config)
train_data = dataset['train']
test_data = dataset['test']

In [ ]:
# Create data loader for training
dataloader = DataLoaderFactory.create_dataloader(dataset=train_data, batch_size=32)

In [ ]:
# Configure device and FGSM attack parameters
device = DeviceConfig(processor="cuda:0")

fgsm_config = FgsmAttackConfig(
    targeted=False,
    epsilon=0.1,
    device=device,
)

# Initialize FGSM attack
fgsm_attack = FGSM(config=fgsm_config)

In [ ]:
# Configure PGD attack parameters
pgd_config = PgdAttackConfig(
    targeted=False,
    epsilon=0.1,
    alpha=0.01,
    num_iter=10,
    device=device,
)

# Initialize PGD attack
pgd_attack = PGD(config=pgd_config)

In [ ]:
# Configure training parameters for FGSM adversarial training
train_config = TrainConfig(
    model_config=ModelConfig(model=model),
    training_process_config=TrainingProcessConfig(
        train_loader=dataloader,
        epochs=20,
        learning_rate=0.001,
        criterion="cross_entropy",
        verbose=True
    ),
    optimization_config=OptimizationBase(optimizer="adam"),
    device_config=device,
    checkpoint_config=CheckpointBase(
        save_checkpoint=True,
        save_checkpoint_path="./checkpoints",
        checkpoint_interval=5
    ),
    final_model_config=FinalModelBase(
        save_final_model=True,
        save_model_path="./models",
        save_model_name="resnet18_cifar10_fgsm_adversarial_trained"
    )
)

print("Adversarial Training Configuration:")
print(f"   Model: ResNet18 with {fc_layer.out_features} classes")
print(f"   Attack: FGSM (epsilon={fgsm_config.epsilon})")
print(f"   Epochs: {train_config.training_process_config.epochs}")
print(f"   Learning Rate: {train_config.training_process_config.learning_rate}")
print(f"   Optimizer: {train_config.optimization_config.optimizer}")
print(f"   Batch Size: {dataloader.batch_size}")
print(f"   Training Samples: {len(dataloader.dataset):,}")

adversarial_training_config = AdversarialTrainingConfig(
    train_config=train_config,
    models=[],
    attacks=[fgsm_attack],
)

adversarial_training = AdversarialTraining(config=adversarial_training_config)

print("\nStarting FGSM adversarial training...")
adversarial_training.train()
print("\nFGSM adversarial training completed!")